# AI Text Detector — BERT Fine-tuning
Run this notebook in **Google Colab** with a GPU runtime:  
`Runtime → Change runtime type → T4 GPU`

## 1. Install dependencies

In [12]:
!pip install -q transformers scikit-learn pandas torch

## 2. Mount Google Drive & load dataset
1. Upload your `dataset.csv` to your Google Drive (e.g. into a folder called `textdetector`)
2. Run this cell — it will ask you to sign in and grant access

In [13]:
from google.colab import drive
drive.mount("/content/drive")

# Update this path to wherever you put dataset.csv in your Drive
DATASET_PATH = "/content/drive/MyDrive/textdetector/dataset.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Imports & GPU check

In [14]:
import pandas as pd
import torch
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA A100-SXM4-80GB


## 4. Load & preprocess data

In [16]:
def preprocess(dataframe):
    print(dataframe.columns.tolist())
    print(dataframe["label"].value_counts())
    print(dataframe.isnull().sum())
    dataframe = dataframe.dropna(subset=["text"])
    dataframe = dataframe.drop_duplicates(subset="text")
    return dataframe

def downsample(dataframe, n=500):
    human = dataframe[dataframe.label == 0]
    ai    = dataframe[dataframe.label == 1]
    human_downsampled = resample(human, replace=False, n_samples=n, random_state=42)
    ai_downsampled    = resample(ai,    replace=False, n_samples=n, random_state=42)
    return pd.concat([human_downsampled, ai_downsampled]).sample(frac=1, random_state=42).reset_index(drop=True)

def split(dataframe):
    train_df, temp_df = train_test_split(
        dataframe, test_size=0.2, random_state=42, stratify=dataframe["label"]
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"]
    )
    return train_df, val_df, test_df

df = pd.read_csv(DATASET_PATH)
df = preprocess(df)
df = downsample(df, n=500)
train_df, val_df, test_df = split(df)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

['title', 'description', 'text', 'label', 'provider']
label
0    2235
1     500
Name: count, dtype: int64
title          0
description    0
text           0
label          0
provider       0
dtype: int64
Train: 800 | Val: 100 | Test: 100


## 5. Dataset class & DataLoaders

In [17]:
class EssayDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.texts     = dataframe["text"].tolist()
        self.labels    = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long)
        }

tokenizer    = BertTokenizer.from_pretrained("bert-base-uncased")
train_loader = DataLoader(EssayDataset(train_df, tokenizer), batch_size=16, shuffle=True)
val_loader   = DataLoader(EssayDataset(val_df,   tokenizer), batch_size=16, shuffle=False)
test_loader  = DataLoader(EssayDataset(test_df,  tokenizer), batch_size=16, shuffle=False)
print("DataLoaders ready.")

DataLoaders ready.


## 6. Load BERT model

In [18]:
model     = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
epochs    = 3
print("Model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.


## 7. Training loop

In [19]:
for epoch in range(1, epochs + 1):
    # --- train ---
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        optimizer.step()
        total_train_loss += outputs.loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # --- validation ---
    model.eval()
    total_val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_val_loss += outputs.loss.item()

            preds    = outputs.logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    avg_val_loss = total_val_loss / len(val_loader)
    val_acc      = correct / total
    print(f"Epoch {epoch}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

Epoch 1/3 | Train Loss: 0.3191 | Val Loss: 0.1226 | Val Acc: 0.9700
Epoch 2/3 | Train Loss: 0.0519 | Val Loss: 0.0236 | Val Acc: 1.0000
Epoch 3/3 | Train Loss: 0.0152 | Val Loss: 0.2477 | Val Acc: 0.9500


## 8. Test set evaluation

In [20]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        preds    = model(input_ids=input_ids, attention_mask=attention_mask).logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

print(f"Test Accuracy: {correct / total:.4f}")

Test Accuracy: 0.9800


## 9. Save the model (optional)
This saves to Colab's temporary storage. Download it before the session ends.

In [ ]:
model.save_pretrained("bert_textdetector")
tokenizer.save_pretrained("bert_textdetector")
print("Model saved to ./bert_textdetector/")

# Download as a zip
import shutil
shutil.make_archive("bert_textdetector", "zip", "bert_textdetector")
files.download("bert_textdetector.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./bert_textdetector/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>